# Práctica Integral — Caso 1: Adquisición de Medicamentos a Demanda

**Asignatura:** Extracción de Conocimiento en Bases de Datos (Unidad II)  
**Equipo:** 1  
**Dataset:** INPRFM — Adquisición de medicamentos demanda (2025)  
**Fuente:** [datos.gob.mx](https://www.datos.gob.mx/dataset/adquisicion_medicamentos_demanda)

Este notebook documenta la fase de Exploración, Diagnóstico de Calidad y Limpieza (E-T-L) del pipeline dimensional.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Rutas
RAW_PATH = '../data/adquisicion_medicamentos.csv'
TRAIN_PATH = '../data/train_dataset.csv'
TEST_PATH = '../data/test_dataset.csv'

# Carga con encoding latino-americano
df = pd.read_csv(RAW_PATH, encoding='latin-1', low_memory=False)
print('Registros crudos:', len(df))
print('Columnas:', df.columns.tolist())

## 1. Diagnóstico de Calidad Inicial

Se inspeccionan: nulos, duplicados, tipos de datos y distribuciones.

In [ ]:
print(df.info())
print('\n--- NULOS ---')
print(df.isnull().sum())
print('\n--- DUPLICADOS ---')
print('Filas duplicadas exactas:', df.duplicated().sum())
print('\n--- DESCRIBE NUMÉRICO ---')
print(df.describe())

## 2. Limpieza Textual Avanzada (Regex)

Se normalizan nombres de proveedores, medicamentos y dependencias para eliminar variantes corporativas, espacios múltiples, minúsculas inconsistentes y caracteres corruptos.

In [ ]:
def limpiar_texto(col):
    col = col.astype(str).str.strip()
    col = col.str.upper()
    col = col.str.replace(r'\s+', ' ', regex=True)
    col = col.str.replace(r'[^A-ZÁÉÍÓÚÑ0-9\s\-/\.]', '', regex=True)
    return col

# Aplicar limpieza a columnas clave
for c in ['cncpt', 'prov', 'sus_activ']:
    df[c] = limpiar_texto(df[c])

# Normalizar fechas (dd/mm/yyyy -> ISO)
df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y', errors='coerce')

# Normalizar numéricos con comas
def to_float(x):
    try:
        return float(str(x).replace(',', '').strip())
    except:
        return np.nan

for c in ['canti', 'prcio_unit', 'total']:
    df[c] = df[c].apply(to_float)

print('Tipos post-limpieza:')
print(df[['fecha','canti','prcio_unit','total']].dtypes)
print('Nulos tras limpieza:', df[['fecha','canti','prcio_unit','total']].isnull().sum().sum())

## 3. Construcción del Esquema Estrella (DataFrames en memoria)

Se descompone la tabla plana en dimensiones y hechos, generando surrogate keys.

In [ ]:
# Dimensión: Tiempo
dim_tiempo = df[['fecha']].drop_duplicates().reset_index(drop=True)
dim_tiempo['tiempo_sk'] = dim_tiempo.index + 1
dim_tiempo['anio'] = dim_tiempo['fecha'].dt.year
dim_tiempo['mes'] = dim_tiempo['fecha'].dt.month
dim_tiempo['dia'] = dim_tiempo['fecha'].dt.day
dim_tiempo['trimestre'] = dim_tiempo['fecha'].dt.quarter

# Dimensión: Medicamento
dim_medicamento = df[['cncpt','sus_activ','forma_farma','cntcn','prese']].drop_duplicates().reset_index(drop=True)
dim_medicamento['medicamento_sk'] = dim_medicamento.index + 1

# Dimensión: Proveedor
dim_proveedor = df[['prov','rfc']].drop_duplicates().reset_index(drop=True)
dim_proveedor['proveedor_sk'] = dim_proveedor.index + 1

# Dimensión: Institución (CLUES)
dim_institucion = df[['clues']].drop_duplicates().reset_index(drop=True)
dim_institucion['institucion_sk'] = dim_institucion.index + 1

# Dimensión: Contratación
dim_contratacion = df[['tipo_adqui','origin_recur']].drop_duplicates().reset_index(drop=True)
dim_contratacion['contratacion_sk'] = dim_contratacion.index + 1

print('Dim Tiempo:', len(dim_tiempo))
print('Dim Medicamento:', len(dim_medicamento))
print('Dim Proveedor:', len(dim_proveedor))
print('Dim Institución:', len(dim_institucion))
print('Dim Contratación:', len(dim_contratacion))

## 4. Tabla de Hechos

Se unen las dimensiones al dataset original mediante los surrogate keys para construir ct_adquisiciones.

In [ ]:
# Merge de surrogate keys
hechos = df.merge(dim_tiempo, on='fecha', how='left')
hechos = hechos.merge(dim_medicamento, on=['cncpt','sus_activ','forma_farma','cntcn','prese'], how='left')
hechos = hechos.merge(dim_proveedor, on=['prov','rfc'], how='left')
hechos = hechos.merge(dim_institucion, on='clues', how='left')
hechos = hechos.merge(dim_contratacion, on=['tipo_adqui','origin_recur'], how='left')

# Tabla de hechos final
fct = hechos[['tiempo_sk','medicamento_sk','proveedor_sk','institucion_sk','contratacion_sk',
              'canti','prcio_unit','total']].copy()

# Variable target predictiva: ¿es adjudicación directa?
hechos['target_adjudicacion_directa'] = (hechos['tipo_adqui'].str.strip().str.upper() == 'ADJUDICACIÓN DIRECTA').astype(int)
print('Distribución del target:')
print(hechos['target_adjudicacion_directa'].value_counts(normalize=True))

# Incorporar target a la tabla de hechos ampliada
fct_full = hechos[['tiempo_sk','medicamento_sk','proveedor_sk','institucion_sk','contratacion_sk',
                   'canti','prcio_unit','total','target_adjudicacion_directa']].copy()
print('Filas tabla hechos:', len(fct_full))

## 5. Validación: Particionamiento Train/Test Balanceado

Se verifica que la proporción del target se conserve en ambos conjuntos (75/25 estratificado).

In [ ]:
X = fct_full.drop(columns=['target_adjudicacion_directa'])
y = fct_full['target_adjudicacion_directa']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print('Train:', len(train_df), '| Test:', len(test_df))
print('\n--- PROPORCIÓN TARGET ---')
print('Original:', y.value_counts(normalize=True).to_dict())
print('Train:   ', y_train.value_counts(normalize=True).to_dict())
print('Test:    ', y_test.value_counts(normalize=True).to_dict())

## 6. Persistencia de Dimensiones (PostgreSQL-ready CSV)

Se exportan las dimensiones limpias para su carga posterior vía carga_dimensiones_hechos.py.

In [ ]:
dim_tiempo.to_csv('../data/dim_tiempo.csv', index=False)
dim_medicamento.to_csv('../data/dim_medicamento.csv', index=False)
dim_proveedor.to_csv('../data/dim_proveedor.csv', index=False)
dim_institucion.to_csv('../data/dim_institucion.csv', index=False)
dim_contratacion.to_csv('../data/dim_contratacion.csv', index=False)
fct_full.to_csv('../data/fct_adquisiciones.csv', index=False)
print('Dimensiones y hechos exportados a ../data/')